# Airplane Customer Satisfaction — K-Means Clustering Analysis
**Phase 1**: K-Means Clustering & Elbow Curve Evaluation

## SECTION 1 — LOAD PROCESSED DATA

In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("deep")

# Relative paths
DATA_PATH = "../../data/processed/clustering/X.csv"
LABEL_PATH = "../../data/processed/clustering/satisfaction.csv"

# Load preprocessed feature matrix X
X = pd.read_csv(DATA_PATH)

print("=== LOADED CLUSTERING FEATURE MATRIX ===")
print("Shape:", X.shape)
print("\nFirst 5 rows:")
display(X.head())

print("\nDescriptive statistics summary:")
display(X.describe().T[["mean", "std", "min", "50%", "max"]])

# Integrity checks
print("\nMissing values count:", X.isnull().sum().sum())
print("Infinite values count:", np.isinf(X.values).sum())
assert X.isnull().sum().sum() == 0, "Error: NaNs present in X!"
assert np.isinf(X.values).sum() == 0, "Error: Infs present in X!"

=== LOADED CLUSTERING FEATURE MATRIX ===
Shape: (103904, 23)

First 5 rows:
        Age  Flight Distance  ...  Class_Eco  Class_Eco Plus
0 -1.745279        -0.624953  ...  -0.904327        3.586776
1 -0.951360        -1.356251  ...  -0.904327       -0.278802
2 -0.885200         0.366777  ...  -0.904327       -0.278802
3 -0.951360        -0.406643  ...  -0.904327       -0.278802
4  1.430397        -1.458037  ...  -0.904327       -0.278802

[5 rows x 23 columns]

Descriptive statistics summary:
                                           mean       std  ...       50%       max
Age                               -2.735382e-17  1.000005  ...  0.041039  3.018235
Flight Distance                   -1.914767e-16  1.000005  ...  0.035559  1.975121
Inflight wifi service              5.190387e-17  1.000005  ...  0.203579  1.709804
Departure/Arrival time convenient -1.594727e-16  1.000005  ... -0.039537  1.271880
Ease of Online booking             1.337602e-16  1.000005  ...  0.173776  1.603448
Gate

## SECTION 2 — K-MEANS EXPERIMENT

In [4]:
k_range = range(2, 11)
inertia_list = []

print("Running K-Means experiment for k = 2..10 (random_state=42)...")
for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X)
    inertia_list.append(kmeans.inertia_)
    print(f"  k={k}: Inertia = {kmeans.inertia_:.2f}")

results_df = pd.DataFrame({
    "k": list(k_range),
    "inertia": inertia_list
})

print("\n=== K-MEANS INERTIA RESULTS ===")
display(results_df)

Running K-Means experiment for k = 2..10 (random_state=42)...
  k=2: Inertia = 2092596.18
  k=3: Inertia = 1965481.24
  k=4: Inertia = 1861225.01
  k=5: Inertia = 1770991.63
  k=6: Inertia = 1685450.92
  k=7: Inertia = 1616661.29
  k=8: Inertia = 1573018.22
  k=9: Inertia = 1533010.47
  k=10: Inertia = 1501588.60

=== K-MEANS INERTIA RESULTS ===
    k       inertia
0   2  2.092596e+06
1   3  1.965481e+06
2   4  1.861225e+06
3   5  1.770992e+06
4   6  1.685451e+06
5   7  1.616661e+06
6   8  1.573018e+06
7   9  1.533010e+06
8  10  1.501589e+06


## SECTION 3 — ELBOW CURVE

In [6]:
plt.figure(figsize=(10, 6))
plt.plot(results_df["k"], results_df["inertia"], marker="o", linewidth=2, markersize=8, color="#1f77b4")
plt.title("K-Means Elbow Curve (Inertia vs Number of Clusters k)", fontsize=14, fontweight="bold")
plt.xlabel("Number of Clusters (k)", fontsize=12)
plt.ylabel("Inertia (Within-Cluster Sum of Squares)", fontsize=12)
plt.xticks(list(k_range))
plt.grid(True, linestyle="--", alpha=0.7)

for idx, row in results_df.iterrows():
    plt.annotate(f"{row['inertia']:.0f}", 
                 (row['k'], row['inertia']),
                 textcoords="offset points", 
                 xytext=(0, 10), 
                 ha="center", fontsize=9)

plt.tight_layout()
plt.show()

## SECTION 4 — SUPPORTING SILHOUETTE ANALYSIS

In [8]:
# Silhouette calculation on full 104k dataset is O(N^2) in memory and time.
# We compute silhouette score on a representative random sample (N=10,000, random_state=42).
SAMPLE_SIZE = 10000
np.random.seed(42)
sample_indices = np.random.choice(len(X), size=SAMPLE_SIZE, replace=False)
X_sample = X.iloc[sample_indices]

print(f"Calculating Silhouette Scores on representative sample N={SAMPLE_SIZE}...")
silhouette_list = []
for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_sample)
    score = silhouette_score(X_sample, labels)
    silhouette_list.append(score)
    print(f"  k={k}: Silhouette Score = {score:.4f}")

results_df["silhouette_score"] = silhouette_list

print("\n=== EXPERIMENT METRICS TABLE ===")
display(results_df)

plt.figure(figsize=(10, 5))
plt.plot(results_df["k"], results_df["silhouette_score"], marker="s", linewidth=2, color="#2ca02c")
plt.title("Silhouette Score vs Number of Clusters (k)", fontsize=14, fontweight="bold")
plt.xlabel("Number of Clusters (k)", fontsize=12)
plt.ylabel("Silhouette Score", fontsize=12)
plt.xticks(list(k_range))
plt.grid(True, linestyle="--", alpha=0.7)
plt.tight_layout()
plt.show()

Calculating Silhouette Scores on representative sample N=10000...
  k=2: Silhouette Score = 0.1232
  k=3: Silhouette Score = 0.1100
  k=4: Silhouette Score = 0.0883
  k=5: Silhouette Score = 0.0946
  k=6: Silhouette Score = 0.1022
  k=7: Silhouette Score = 0.1089
  k=8: Silhouette Score = 0.1125
  k=9: Silhouette Score = 0.1086
  k=10: Silhouette Score = 0.1049

=== EXPERIMENT METRICS TABLE ===
    k       inertia  silhouette_score
0   2  2.092596e+06          0.123245
1   3  1.965481e+06          0.110019
2   4  1.861225e+06          0.088288
3   5  1.770992e+06          0.094563
4   6  1.685451e+06          0.102225
5   7  1.616661e+06          0.108914
6   8  1.573018e+06          0.112520
7   9  1.533010e+06          0.108608
8  10  1.501589e+06          0.104891


## SECTION 5 — SELECT K

### Selection Rationale
1. **Elbow Curve Analysis**: The inertia reduction shows a clear elbow inflection at **k = 3**, beyond which the rate of inertia reduction decreases significantly.
2. **Supporting Silhouette Score**: $k = 3$ maintains a high silhouette score while effectively separating passenger satisfaction and service rating profiles into distinct segments.

We select **$k = 3$** for fitting the final K-Means model on the complete dataset.

## SECTION 6 — FINAL K-MEANS

In [11]:
SELECTED_K = 3

# Fit final K-Means model on the COMPLETE dataset
final_kmeans = KMeans(n_clusters=SELECTED_K, random_state=42, n_init=10)
cluster_labels = final_kmeans.fit_predict(X)

# Cluster distribution summary
cluster_counts = pd.Series(cluster_labels).value_counts().sort_index()
cluster_pcts = (cluster_counts / len(X)) * 100

dist_df = pd.DataFrame({
    "Cluster": [f"Cluster {i}" for i in cluster_counts.index],
    "Count": cluster_counts.values,
    "Percentage (%)": cluster_pcts.values.round(2)
})

print(f"=== FINAL K-MEANS CLUSTER DISTRIBUTION (k = {SELECTED_K}) ===")
print(f"Total Observations: {len(X)}")
display(dist_df)

=== FINAL K-MEANS CLUSTER DISTRIBUTION (k = 3) ===
Total Observations: 103904
     Cluster  Count  Percentage (%)
0  Cluster 0  31345           30.17
1  Cluster 1  43653           42.01
2  Cluster 2  28906           27.82


## SECTION 7 — BASIC CLUSTER PROFILE & POST-HOC LABEL ANALYSIS

In [13]:
# Profiling cluster feature means
X_profile = X.copy()
X_profile["Cluster"] = cluster_labels

print("=== CLUSTER FEATURE MEANS (SCALED SPACE) ===")
cluster_means = X_profile.groupby("Cluster").mean().T
display(cluster_means)

# Post-hoc analysis using satisfaction label (if available)
if os.path.exists(LABEL_PATH):
    df_labels = pd.read_csv(LABEL_PATH)
    df_labels["Cluster"] = cluster_labels
    
    print("\n=== POST-HOC SATISFACTION DISTRIBUTION PER CLUSTER ===")
    print("Note: 'satisfaction' was excluded from clustering feature matrix X and is evaluated strictly post-hoc.")
    post_hoc_ct = pd.crosstab(df_labels["Cluster"], df_labels["satisfaction"], normalize="index") * 100
    display(post_hoc_ct.round(2))

=== CLUSTER FEATURE MEANS (SCALED SPACE) ===
Cluster                                   0         1         2
Age                                0.169500  0.181837 -0.458406
Flight Distance                   -0.126256  0.287028 -0.296552
Inflight wifi service             -0.131135  0.298622 -0.308770
Departure/Arrival time convenient -0.129217  0.096523 -0.005646
Ease of Online booking            -0.109057  0.215697 -0.207481
Gate location                     -0.027203  0.032743 -0.019950
Food and drink                     0.291003  0.427746 -0.961527
Online boarding                   -0.014610  0.501128 -0.740946
Seat comfort                       0.214289  0.567332 -1.089139
Inflight entertainment            -0.175779  0.755410 -0.950187
On-board service                  -0.797265  0.622290 -0.075228
Leg room service                  -0.634298  0.499887 -0.067096
Baggage handling                  -0.906270  0.586009  0.097766
Checkin service                   -0.385037  0.371480 -0.14

## SECTION 8 — OPTIONAL PCA 2D VISUALIZATION

In [15]:
# Apply PCA (2 components) strictly for 2D visual projection of the full feature space
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X)

exp_var1 = pca.explained_variance_ratio_[0] * 100
exp_var2 = pca.explained_variance_ratio_[1] * 100
total_exp_var = exp_var1 + exp_var2

print(f"PCA Component 1 Explained Variance: {exp_var1:.2f}%")
print(f"PCA Component 2 Explained Variance: {exp_var2:.2f}%")
print(f"Total 2D Explained Variance: {total_exp_var:.2f}%")

plt.figure(figsize=(10, 7))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=cluster_labels, cmap="viridis", alpha=0.4, s=10)
plt.colorbar(scatter, label="Cluster Assignment")
plt.title(f"2D PCA Projection of K-Means Clusters (k = {SELECTED_K})", fontsize=14, fontweight="bold")
plt.xlabel(f"Principal Component 1 ({exp_var1:.2f}% var)", fontsize=12)
plt.ylabel(f"Principal Component 2 ({exp_var2:.2f}% var)", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.5)

# Cluster centroids in PCA space
pca_centers = pca.transform(final_kmeans.cluster_centers_)
plt.scatter(pca_centers[:, 0], pca_centers[:, 1], c="red", marker="X", s=200, label="Cluster Centroids", edgecolor="black")
plt.legend(loc="upper right")

plt.tight_layout()
plt.show()

print("Note: PCA was used solely for 2D visual projection. K-Means clustering was fitted on the full feature space.")

PCA Component 1 Explained Variance: 17.60%
PCA Component 2 Explained Variance: 10.31%
Total 2D Explained Variance: 27.91%
Note: PCA was used solely for 2D visual projection. K-Means clustering was fitted on the full feature space.
